# 02 Train Models

Thin training runner. It reads the latest dataset config, constructs models from `state_dim`, trains baseline and stable dynamics models, and saves checkpoints plus `latest_training_config.json`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / "stable_icnn_physics").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

for module_name in list(sys.modules):
    if module_name == "stable_icnn_physics" or module_name.startswith("stable_icnn_physics."):
        del sys.modules[module_name]

import json
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyTorch is not installed in this notebook kernel. Install dependencies or switch kernels.") from exc

from stable_icnn_physics import BaselineDynamicsMLP, build_stable_model
from stable_icnn_physics.data import load_dataset, tensor_dataset
from stable_icnn_physics.plotting import plot_loss
from stable_icnn_physics.train import evaluate_derivative_mse, train_derivative_model

CACHE_DIR = REPO_ROOT / "data" / "cache"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
CONFIG_PATH = CACHE_DIR / "latest_dataset_config.json"
TRAINING_CONFIG_PATH = CHECKPOINT_DIR / "latest_training_config.json"

EPOCHS = 200
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
HIDDEN = 100
DEPTH = 2
LYAPUNOV_HIDDEN = 60
LYAPUNOV_EPS = 0.01
ALPHA = 1e-3
REHU_WIDTH = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing {CONFIG_PATH}. Run 01_generate_data.ipynb first.")
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
state_dim = int(config["system"]["state_dim"])
system_name = config["system"]["name"]
dataset_mode = config["dataset_mode"]
print("system:", system_name, "state_dim:", state_dim, "dataset_mode:", dataset_mode, "device:", DEVICE)


## Load Dataset

In [ ]:
train_dataset_path = Path(config["train_dataset_path"])
test_dataset_path = Path(config["test_dataset_path"])
if not train_dataset_path.exists() or not test_dataset_path.exists():
    raise FileNotFoundError("Dataset files referenced by latest_dataset_config.json were not found.")

x_train, y_train = load_dataset(train_dataset_path)
x_test, y_test = load_dataset(test_dataset_path)
train_ds = tensor_dataset(x_train, y_train)
test_ds = tensor_dataset(x_test, y_test)

print("train:", x_train.shape, y_train.shape)
print("test: ", x_test.shape, y_test.shape)
print("x mean/std:", x_train.mean(axis=0), x_train.std(axis=0))
print("y mean/std:", y_train.mean(axis=0), y_train.std(axis=0))


## Quick Data Check

In [ ]:
sample_count = min(1200, len(x_train))
rng = np.random.default_rng(config.get("seed", 0))
idx = rng.choice(len(x_train), size=sample_count, replace=False)

cols = min(state_dim, 4)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x_train[idx, 0], x_train[idx, 1], s=8, alpha=0.3)
axes[0].set_xlabel("state[0]")
axes[0].set_ylabel("state[1]")
axes[0].set_title("Train states")
axes[0].grid(alpha=0.25)
for j in range(cols):
    axes[1].hist(y_train[:, j], bins=50, alpha=0.45, label=f"ydot[{j}]")
axes[1].set_title("Target component histograms")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()


## Build Models

In [ ]:
stable_model = build_stable_model(
    dim=state_dim,
    hidden=HIDDEN,
    depth=DEPTH,
    lyapunov_hidden=LYAPUNOV_HIDDEN,
    lyapunov_eps=LYAPUNOV_EPS,
    alpha=ALPHA,
    rehu_width=REHU_WIDTH,
)
baseline_model = BaselineDynamicsMLP(dim=state_dim, hidden=HIDDEN, depth=DEPTH)

print("stable parameters:", sum(p.numel() for p in stable_model.parameters()))
print("baseline parameters:", sum(p.numel() for p in baseline_model.parameters()))


## Train

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
stem = f"{system_name}_{dataset_mode}"
stable_ckpt_path = CHECKPOINT_DIR / f"{stem}_stable.pt"
baseline_ckpt_path = CHECKPOINT_DIR / f"{stem}_baseline.pt"

stable_history = train_derivative_model(
    stable_model, train_ds, test_ds,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    device=DEVICE,
    checkpoint_path=stable_ckpt_path,
    print_every=max(1, EPOCHS // 8),
)

baseline_history = train_derivative_model(
    baseline_model, train_ds, test_ds,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    device=DEVICE,
    checkpoint_path=baseline_ckpt_path,
    print_every=max(1, EPOCHS // 8),
)


## Diagnostics And Training Config

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_loss(stable_history, ax=axes[0], title="Stable model")
plot_loss(baseline_history, ax=axes[1], title="Baseline MLP")
axes[2].plot(stable_history.test_loss, label="stable test")
axes[2].plot(baseline_history.test_loss, label="baseline test")
axes[2].set_yscale("log")
axes[2].set_xlabel("epoch")
axes[2].set_ylabel("MSE")
axes[2].set_title("Test loss comparison")
axes[2].legend()
axes[2].grid(alpha=0.25)
plt.tight_layout()

stable_test_mse = evaluate_derivative_mse(stable_model, test_ds, device=DEVICE)
baseline_test_mse = evaluate_derivative_mse(baseline_model, test_ds, device=DEVICE)
print("stable test MSE:  ", stable_test_mse)
print("baseline test MSE:", baseline_test_mse)

training_config = {
    "stable_checkpoint_path": str(stable_ckpt_path),
    "baseline_checkpoint_path": str(baseline_ckpt_path),
    "dataset_config_path": str(CONFIG_PATH),
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "hidden": HIDDEN,
        "depth": DEPTH,
        "lyapunov_hidden": LYAPUNOV_HIDDEN,
        "lyapunov_eps": LYAPUNOV_EPS,
        "alpha": ALPHA,
        "rehu_width": REHU_WIDTH,
    },
    "state_dim": state_dim,
    "system": config["system"],
    "dataset_mode": dataset_mode,
    "metrics": {
        "stable_test_mse": float(stable_test_mse),
        "baseline_test_mse": float(baseline_test_mse),
    },
}
TRAINING_CONFIG_PATH.write_text(json.dumps(training_config, indent=2), encoding="utf-8")
print("stable checkpoint:  ", stable_ckpt_path)
print("baseline checkpoint:", baseline_ckpt_path)
print("training config:    ", TRAINING_CONFIG_PATH)
